In [ ]:
%load_ext autoreload
%autoreload 2

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scripts.TPS import ThinPlateSpline
from scripts.plotting import plot_velocity_streamplot

# ---------- Load vector field ----------
def load_vector_field(csv_path):
    df = pd.read_csv(csv_path)
    X = df[["x", "y"]].values
    V = df[["vx", "vy"]].values
    time = df["time"].values
    return X, V, time

# ---------- File organization ----------
path_map = {
    "straight_line": "./data/1d/straight_line.csv",
    "sine_curve": "./data/1d/sine_curve.csv",
    "branch_2": "./data/1d/branch_2.csv",
    "branch_4": "./data/1d/branch_4.csv",
    "rotation": "./data/2d/rotation.csv",
    "spiral": "./data/2d/spiral.csv",
    "saddle": "./data/2d/saddle.csv",
    "quadratic_source_sink": "./data/2d/quadratic_source_sink.csv"
}

plot_order = [
    "straight_line", "sine_curve", "branch_2", "branch_4",
    "rotation", "spiral", "saddle", "quadratic_source_sink"
]

# ---------- Plot 1×8 row ----------
fig, axs = plt.subplots(1, 8, figsize=(32, 4))

for i, (ax, name) in enumerate(zip(axs, plot_order)):
    X, V, time = load_vector_field(path_map[name])
    time = (time - np.min(time)) / (np.max(time) - np.min(time))  # Normalize per plot

    tps_vf = ThinPlateSpline(X, n_control_points=100)
    tps_vf.fit(V, dof=15)

    # Per-panel stream density and aspect ratio
    if name == "straight_line":
        stream_density = 0.2
        aspect = 2.5
    elif name == "sine_curve":
        stream_density = 0.4
        aspect = 2.0
    elif name in {"branch_2", "branch_4"}:
        stream_density = 0.4
        aspect = 1.5
    else:
        stream_density = 0.5
        aspect = "equal"

    plot_velocity_streamplot(
        X_2d=X,
        tps_vf=tps_vf,
        grid_density=1.0, 
        stream_density=stream_density,
        scatter_color=time,
        scatter_size=800,
        scatter_alpha=0.05,
        ax=ax,
        title=None,  # <- No title
        aspect=aspect,
        vmin=0.0,
        vmax=1.0,
        cmap="viridis",
        show_axes=False,
        arrowsize=3.0,
        streamline_thickness=4.0,
        grid_size=50
    )

plt.tight_layout()
plt.show()

In [ ]:
np.random.seed(42)
simulation_results = {}
noise = 0.2
extra_dim = 2

for i, (name, path) in enumerate(path_map.items()):
    # Load clean 2D data
    X_gt, V_gt, time = load_vector_field(path_map[name])

    # Add noise
    X_noisy = X_gt + np.random.normal(scale=noise, size=X_gt.shape)
    V_noisy = V_gt + np.random.normal(scale=noise, size=V_gt.shape)

    # Add dummy dimensions
    X_dummy = np.random.normal(scale=noise, size=(X_gt.shape[0], extra_dim))
    V_dummy = np.random.normal(scale=noise, size=(V_gt.shape[0], extra_dim))

    X = np.hstack([X_noisy, X_dummy])
    V = np.hstack([V_noisy, V_dummy])
    
    # Save all results
    simulation_results[name] = {
        "X": X,
        "V": V,
        "true_time": time
    }

In [ ]:
from scripts.VectorFieldEmbedder import *

np.random.seed(42)
embedding_results = {}

for name, data in simulation_results.items():
    X = data["X"]
    V = data["V"]
    time = data["true_time"]
    time = (time - np.min(time)) / (np.max(time) - np.min(time))  # normalize

    emb = VectorFieldEmbedder(
        X, V, use_PCA=False,
        embed_kwargs={"n_neighbors": 30, "min_dist": 0.3}
    )
    emb.create_chart()
    emb.optimize()

    embedding_results[name] = {
        "embedder": emb,
        "time": time
    }

In [ ]:
fig, axs = plt.subplots(1, 8, figsize=(32, 4))

for ax, name in zip(axs, embedding_results.keys()):
    emb = embedding_results[name]["embedder"]
    time = embedding_results[name]["time"]
    
    # Per-panel stream density and aspect ratio
    if name == "straight_line":
        stream_density = 0.3
        aspect = 2.5
    elif name == "sine_curve":
        stream_density = 0.5
        aspect = 2.0
    elif name in {"branch_2", "branch_4"}:
        stream_density = 0.6
        aspect = 1.5
    else:
        stream_density = 0.6
        aspect = "equal"
    
    plot_velocity_streamplot(
        X_2d=emb.X_emb_init,  # Or emb.chart if you want the optimized one
        tps_vf=emb.tps_viz_vf,
        grid_density=1.0, 
        stream_density=stream_density,
        scatter_color=time,
        scatter_size=800,
        scatter_alpha=0.1,
        ax=ax,
        title=None,  # <- No title
        aspect=aspect,
        vmin=0.0,
        vmax=1.0,
        arrowsize=3.0,
        cmap="viridis",
        show_axes=False,
        streamline_thickness=4.0,
        grid_size=50
    )

plt.tight_layout()
plt.show()